This is a fantastic idea for a class. Real-world engineering data is infinitely more engaging for students and interns than generic "Titanic" or "Iris" datasets. Hydrology data is inherently messy, which makes it the perfect sandbox for teaching `pandas`.

For a 2-hour timeframe, we should focus on high-impact, frequently used functions. I recommend using **"46543000_Vazoes.csv"** (Time Series) and **"46543000_ResumoDescarga.csv"** (Field Measurements). 

Here is a structured lesson plan designed to take engineering students from raw data to a visual "dashboard" in 120 minutes.

### **The Problem Statement (The Hook)**
**"The Hydrological Snapshot"**
> *"We have been tasked with analyzing the historical behavior of the Rio de Ondas (Station 46543000). To deliver our preliminary report, we must answer two questions using data: 1) What does the historical flow look like, and what are the yearly maximums? 2) How does the physical depth of the river (Cota) translate to water volume (Vazão)? We will build a 2-chart interactive dashboard to present our findings."*

---

### **Class Structure (120 Minutes)**

#### **Part 1: Ingestion & The Messy Reality (30 mins)**
*Goal: Teach students how to load real, messy CSVs and clean them.*
* **The Challenge:** ANA Hidroweb files have metadata headers (skiprows), use semicolons (sep), and have commas for decimals (decimal/replace).
* **Data Used:** "46543000_Vazoes.csv" and "46543000_ResumoDescarga.csv"
* **Key Pandas Functions:** 
    * `pd.read_csv(..., sep=';', skiprows=13, encoding='latin-1')`
    * `.head()`, `.info()`, `.columns`
    * `.dropna()` (Removing empty rows from the raw export)
    * **String manipulation:** Converting the string floats to actual floats (e.g., `df['Vazao'].str.replace(',', '.').astype(float)`).

#### **Part 2: Time Travel & Aggregation (35 mins)**
*Goal: Master dates, filtering, and grouped math (crucial for time-series engineering data).*
* **The Challenge:** We have daily flow data, but we want to see the annual maximums and calculate the long-term monthly averages to spot wet/dry seasons.
* **Data Used:** "46543000_Vazoes.csv"
* **Key Pandas Functions:**
    * `pd.to_datetime(..., format='%d/%m/%Y')` (Creating a proper time index).
    * `.dt.year` and `.dt.month` (Extracting features).
    * Boolean Indexing: Filtering out bad data (`df[df['NivelConsistencia'] == 2]`).
    * `.groupby('Ano')['MediaDiaria'].max()` (Finding the peak flood per year).

#### **Part 3: Visualizing with Plotly Express (35 mins)**
*Goal: Turn tables into interactive engineering tools.*
* **The Challenge:** Creating intuitive plots that allow users to hover over exact dates and values.
* **Data Used:** Both cleaned datasets.
* **Key Plotly Functions:**
    * `px.line()`: Plotting the continuous time-series of "MediaDiaria" over "Data" from the flow dataset.
    * `px.scatter()`: Plotting "Vazao" vs. "Cota" from the "46543000_ResumoDescarga.csv" file to visualize the empirical rating curve (Curva-Chave).
    * `.update_layout()`: Adding titles and axis labels.

#### **Part 4: The Final Dashboard (20 mins)**
*Goal: Combine the insights into a single deliverable.*
* **The Challenge:** Putting the time-series and the scatter plot side-by-side using `make_subplots`. 
* **Key Plotly Functions:**
    * `make_subplots(rows=1, cols=2)`
    * `fig.add_trace()`
    * `fig.show()` or `fig.write_html("dashboard_rio_ondas.html")` (Showing them how to export their work as a standalone interactive file they can send to a manager).

---
Would you like me to draft a quick "student handout" with step-by-step coding challenges and blanks for them to fill in during the class?


In [ ]:
import pandas as pd
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# 1. Load and Clean
vazoes = pd.read_csv("46543000_Vazoes.csv", sep=";", skiprows=13, encoding="latin-1", index_col=False)
vazoes['Data'] = pd.to_datetime(vazoes['Data'], format='%d/%m/%Y')
vazoes = vazoes.dropna(subset=['MediaDiaria'])

descarga = pd.read_csv("46543000_ResumoDescarga.csv", sep=";", skiprows=13, encoding="latin-1", index_col=False)
for col in ["Cota", "Vazao"]:
    descarga[col] = descarga[col].astype(str).str.replace(",", ".").astype(float)
descarga = descarga.dropna(subset=['Cota', 'Vazao'])

# 2. Build Dashboard
fig = make_subplots(rows=1, cols=2, subplot_titles=("Série Histórica de Vazão", "Curva de Descarga (Cota x Vazão)"))

# Plot 1: Time Series
fig.add_trace(go.Scatter(x=vazoes['Data'], y=vazoes['MediaDiaria'], mode='lines', name='Vazão Diária'), row=1, col=1)

# Plot 2: Scatter
fig.add_trace(go.Scatter(x=descarga['Vazao'], y=descarga['Cota'], mode='markers', name='Medições em Campo'), row=1, col=2)

fig.update_layout(height=500, width=1000, title_text="Dashboard Hidrológico: Estação 46543000")
fig.show()
